Cell 1: Setup & Connection
First, authenticate and connect DuckDB to the Hugging Face warehouse. Note: You must paste your own read token where indicated.

In [3]:
!pip install duckdb pandas scikit-learn huggingface_hub

import duckdb
import pandas as pd
import getpass
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_score

# Securely input your Hugging Face READ token
print("Enter your Hugging Face READ Token (starts with hf_):")
HF_TOKEN = getpass.getpass()

# Connect DuckDB and authenticate
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# Verify connection by checking the client table
print("Connection successful. Verifying tables...")
con.sql(f"SELECT COUNT(*) as client_count FROM read_parquet('{REL}/dim_clients.parquet')").show()

Enter your Hugging Face READ Token (starts with hf_):
··········
Connection successful. Verifying tables...
┌──────────────┐
│ client_count │
│    int64     │
├──────────────┤
│          104 │
└──────────────┘



Cell 2: Feature Engineering & Target Definition
This query does the heavy lifting. It looks at 60 days of historical data to create features (like impression changes and CTR variance) and checks the subsequent 30 days to create the binary label (1 if impressions dropped by more than 20%, 0 otherwise).

In [6]:
# Use DuckDB to crunch the daily facts and build a feature table
query = f"""
WITH content_history AS (
    SELECT
        content_hash_id,
        SUM(CASE WHEN report_date >= '2026-03-01' AND report_date < '2026-04-01' THEN gsc_impressions ELSE 0 END) as impressions_period_1,
        SUM(CASE WHEN report_date >= '2026-04-01' AND report_date < '2026-05-01' THEN gsc_impressions ELSE 0 END) as impressions_period_2,

        -- Calculate CTR dynamically: SUM(clicks) / SUM(impressions)
        CAST(SUM(CASE WHEN report_date >= '2026-03-01' AND report_date < '2026-05-01' THEN gsc_clicks ELSE 0 END) AS FLOAT) /
        NULLIF(SUM(CASE WHEN report_date >= '2026-03-01' AND report_date < '2026-05-01' THEN gsc_impressions ELSE 0 END), 0) as avg_historical_ctr,

        SUM(CASE WHEN report_date >= '2026-05-01' AND report_date < '2026-06-01' THEN gsc_impressions ELSE 0 END) as target_impressions
    FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date >= '2026-03-01' AND report_date < '2026-06-01'
    GROUP BY content_hash_id
)
SELECT
    content_hash_id,
    impressions_period_1,
    impressions_period_2,
    avg_historical_ctr,
    -- Feature: Momentum (Are impressions already decaying?)
    CAST(impressions_period_2 AS FLOAT) / NULLIF(impressions_period_1, 0) as momentum_ratio,
    target_impressions,
    -- Label: Did impressions drop by >20% in the target month?
    CASE
        WHEN target_impressions < (impressions_period_2 * 0.8) THEN 1
        ELSE 0
    END as is_decaying
FROM content_history
WHERE impressions_period_1 > 50 -- Filter out extremely low-volume noise
"""

print("Executing DuckDB Query (this may take 60-90 seconds over the Parquet files)...")
df = con.sql(query).df()
print(f"Dataframe loaded with {len(df)} rows.")
df.dropna(inplace=True)
display(df.head())

Executing DuckDB Query (this may take 60-90 seconds over the Parquet files)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataframe loaded with 115723 rows.


,content_hash_id,impressions_period_1,impressions_period_2,avg_historical_ctr,momentum_ratio,target_impressions,is_decaying
0,content_b7e512995f79d5a6,1140.0,1151.0,0.001746,1.009649,878.0,1
1,content_05597932fe4da067,57.0,73.0,0.000000,1.280702,22.0,1
2,content_905aa32a0230694e,149.0,98.0,0.000000,0.657718,167.0,0
3,content_05434271b257bb68,1421.0,2275.0,0.009740,1.600985,1783.0,1
4,content_d056587ff7faca0c,2770.0,6266.0,0.002435,2.262094,6684.0,0


Cell 3: Baseline Comparison & Model Training
This section trains the model and outputs the exact metrics you need for the "Results" section of your research paper.

In [7]:
# 1. Define Features (X) and Target (y)
features = ['impressions_period_1', 'impressions_period_2', 'avg_historical_ctr', 'momentum_ratio']
X = df[features]
y = df['is_decaying']

# 2. Time-Aware/Standard Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Create a Hardcoded Baseline Rule
# (e.g., predict decay if momentum is already dropping and CTR is low)
df_test = df.loc[X_test.index].copy()
df_test['baseline_pred'] = ((df_test['momentum_ratio'] < 0.9) & (df_test['avg_historical_ctr'] < 0.03)).astype(int)
baseline_precision = precision_score(y_test, df_test['baseline_pred'])

# 4. Train the Machine Learning Model
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

# 5. Evaluate the Model
y_pred = model.predict(X_test)
model_precision = precision_score(y_test, y_pred)

print("--- CAPSTONE RESULTS FOR YOUR PAPER ---")
print(f"Baseline Heuristic Precision: {baseline_precision:.2f}")
print(f"ML Model Precision: {model_precision:.2f}")
print("\nFull Classification Report:")
print(classification_report(y_test, y_pred))

print("\nFeature Importances:")
for name, importance in zip(features, model.feature_importances_):
    print(f"{name}: {importance:.3f}")

--- CAPSTONE RESULTS FOR YOUR PAPER ---
Baseline Heuristic Precision: 0.56
ML Model Precision: 0.64

Full Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.60      0.57     10440
           1       0.64      0.57      0.60     12705

    accuracy                           0.58     23145
   macro avg       0.59      0.59      0.58     23145
weighted avg       0.59      0.58      0.59     23145


Feature Importances:
impressions_period_1: 0.053
impressions_period_2: 0.293
avg_historical_ctr: 0.436
momentum_ratio: 0.218


Cell 4: Generating Ranked Recommendations
This final cell creates the action playbook. It scores the pages and outputs a prioritized list that content teams can use to refresh decaying articles.

In [8]:
# Generate decay probability scores for the test set
df_test['decay_probability'] = model.predict_proba(X_test)[:, 1]

# Sort by the highest risk of decay to create the ranked queue
ranked_queue = df_test.sort_values(by='decay_probability', ascending=False)

# Categorize into the Action Playbook thresholds
def assign_action(score):
    if score >= 0.80:
        return "Rewrite Immediately (Critical Risk)"
    elif score >= 0.50:
        return "Improve Metadata / Refresh (Warning)"
    else:
        return "Monitor & Protect (Stable)"

ranked_queue['recommended_action'] = ranked_queue['decay_probability'].apply(assign_action)

print("--- RANKED ACTION QUEUE (TOP 10 AT-RISK PAGES) ---")
display(ranked_queue[['content_hash_id', 'decay_probability', 'recommended_action']].head(10))

--- RANKED ACTION QUEUE (TOP 10 AT-RISK PAGES) ---


,content_hash_id,decay_probability,recommended_action
14762,content_a2f41bc0f7f9dce1,0.729871,Improve Metadata / Refresh (Warning)
14839,content_7a6796cca25581eb,0.729871,Improve Metadata / Refresh (Warning)
15054,content_3c93551343eb9e23,0.729871,Improve Metadata / Refresh (Warning)
14697,content_a2ed15998248c32b,0.729871,Improve Metadata / Refresh (Warning)
72900,content_07e624e95f185c37,0.729871,Improve Metadata / Refresh (Warning)
72826,content_8912d97a9362d147,0.729871,Improve Metadata / Refresh (Warning)
73192,content_7f10c37b4117f8f6,0.729871,Improve Metadata / Refresh (Warning)
14971,content_2c365fbbcf36c84f,0.729871,Improve Metadata / Refresh (Warning)
73105,content_533fb20321daea4f,0.729871,Improve Metadata / Refresh (Warning)
72751,content_dfb4e04db6042d30,0.729871,Improve Metadata / Refresh (Warning)
